In [1]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

from examples.seismic import Model, plot_velocity, TimeAxis, RickerSource, Receiver
from devito import TimeFunction, VectorTimeFunction, TensorTimeFunction, Eq, solve, Operator, NODE
from devito.finite_differences.operators import div, grad
from matplotlib.animation import FuncAnimation

from lista_utils import *

from examples.seismic.stiffness.utils import C_Matrix, D, S, vec

# QUESTÃO 3

In [ ]:
# Construindo modelo de velocidade

nx, nz = 2001, 151 # Quantidade de pontos nas direções X e Z (151 pontos)
dx, dz = 2.5, 2.5 # Espaçamento entre pontos nas direções X e Z (10 metros)
origin = (0., 0.)  # Coordenadas da origem do modelo
dtype = 'float32'
nbl = 50
space_order = 8

vp = np.ones((nx,nz), dtype=dtype) * 2
vs = vp / 2 * 0.
rho = vp
b = 1 / rho

model = Model(vp=vp, vs=vs, b=b, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs='damp')

# Plotando o modelo de velocidade

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet', 'vmin':model.mu.data.min(), 'vmax':model.lam.data.max()}

fig, axes = plt.subplots(1, 3, figsize=(15,4))

img = axes[0].imshow(model.lam.data.T, **plot_options)
axes[0].set_title('$\\lambda$')
axes[0].set_xlabel('Distância (m)')
axes[0].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

img = axes[1].imshow(model.mu.data.T, **plot_options)
axes[1].set_title('$\\mu$')
axes[1].set_xlabel('Distância (m)')
axes[1].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

img = axes[2].imshow(1/model.b.data.T, **plot_options)
axes[2].set_title('$\\rho$')
axes[2].set_xlabel('Distância (m)')
axes[2].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

fig.tight_layout()
plt.show()

In [ ]:
# Definindo a geometria de aquisição (fontes e receptores)

t0 = 0.  # Tempo inicial da modelagem t=0ms
tn = 2000.  # Tempo final da modelagem t=1000ms
dt = model.critical_dt  # Tempo entre iterações (2ms)
f0 = 0.01  # Frequência de pico da wavelet (20Hz = 0.020 kHz)
ns = 1 # Número de tiros
ng = nx # Número de receptores

s_pos = np.zeros((ns,2)) + np.asarray(model.domain_size) * 0.5 # Posição das fontes

g_pos = np.zeros((ng,2)) # Posição dos receptores
g_pos[:, 0] = np.linspace(0, model.domain_size[0], ng) # Posição dos receptores (x)
g_pos[:, 1] = 0 # Posição dos receptores (z)

src, rec = get_src_rec(model, t0, tn, dt, f0, ns, ng, s_pos, g_pos, order=1)

# Plotando aquisição
plot_aquisition_setup(model, src, rec[0], param='lam')

Partindo da equação da onda elástica de primeira ordem:

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-{\bf D} \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-{\bf C D}^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}

onde ${\bf C}$ é o tensor elástico isotrópico na notação de Voigt e ${\bf D}$ é uma coleção de operadores diferenciais, definidos como

\begin{equation}
\begin{split}
   {\bf C}=\left(\begin{array}{ccc}
   \lambda+2 \mu & \lambda & 0 \\
   \lambda & \lambda+2 \mu & 0 \\ 
   0 & 0 & \mu \end{array}\right)~~\text{e}~~
{\bf D}=\left(\begin{array}{ccc}  
\dfrac{\partial}{\partial x}& 0 &\dfrac{\partial}{\partial z} \\
0 &  \dfrac{\partial}{\partial z}&\dfrac{\partial}{\partial x}
\end{array}\right)  .
\end{split}
\end{equation}

In [ ]:
# Plotando sismogramas

V, sigma, rec = elastic_forward(model, src, rec, order=1)

amax = max(rec[2].data.max(), abs(rec[2].data.min())) * .5
plot_options = {'extent':[0, nx * dx, src.time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto', 'vmin':-amax, 'vmax':amax}

fig, ax = plt.subplots(figsize=(5,7))

ax.imshow(rec[2].data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

## a)

In [5]:
# Plotando snapshots

# amax = max(abs(sigma[0].data.min()), sigma[0].data.max()) / 1000
# plot_snaps(sigma[0], model, src, 100, 2000, 20, cols=6, vmin_max=[-amax, amax])

## b)

In [6]:
# Plotando filmagem da propagação

# OBS01: ESTA CÉLULA DEMORA MUITO PARA EXECUTAR
# OBS02: PARA EXECUTAR O VÍDEO, BASTA DAR PLAY NO WIDGET QUE APARECERÁ

# output = plot_video(sigma[0], rec_sigma, model, interval=1, factor=1000)

# output

## c)

A amplitude da onda diminui com o passar do tempo devido a três principais fatores: divergência esférica, reflexões e amortecimento da borda. O primeiro fator é um fator físico e acontece na realidade. A perda de energia por divergência esférica se dá devido à propagação radial/esférica da onda. A energia inicial da onda, concentrada em aproximadamente um ponto, é a única responsável por sua propagação (energia total) e à medida em que a onda se propaga essa mesma energia, antes concentrada em um ponto, deve se dividir em uma frente de onda esférica cujo raio aumenta com o tempo.

De acordo com a equação da área da superfície de uma esfera

$$ A=4 \pi R^2 $$

podemos dizer que a energia da onda diminui com o quadrado do raio.

Outro fator impactante para a diminuição da amplitude na modelagem é o amortecimento da borda. Em casos de simulação computacional, deve ser adicionada ao modelo uma borda atenuante para que a simulação se torne mais coerente com a realidade (modelo infinito), afinal na Terra não há bordas. Portanto, grande parte da atenuação da onda nessa modelagem se deu devido à borda atenuante.

# Questão 5

## a)

In [ ]:
plot_options = {'extent':[0, nx * dx, src.time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto'}

fig, axes = plt.subplots(1, 2, figsize=(8,7), sharey=True)

amax = max(rec[1].data.max(), abs(rec[1].data.min())) * .5
axes[0].imshow(rec[1].data, **plot_options, vmax=rec[1].data.max() * 0.5)
axes[0].set_title('$v_{zz}$')
axes[0].set_xlabel('Distância (m)')

amax = max(rec[2].data.max(), abs(rec[2].data.min())) * .5
axes[1].imshow(rec[2].data, **plot_options, vmax=rec[2].data.max() * 0.5)
axes[1].set_title('$\\sigma$')
axes[1].set_xlabel('Distância (m)')

fig.tight_layout()
plt.show()

## b) e c)

A diferença entre os registros da componente vertical da velocidade de partícula $v_{zz}$ e da pressão $\sigma$ é baixa. Isso ocorre porque, na superfície, onde estão os receptores, a frente de onda caminha em uma direção próxima da vertical. Assim, a maior parte da energia da onda está sendo usada para deslocar as partículas nessa direção, como é observado no sismograma obtido da componente $v_{zz}$

# Questão 6

In [ ]:
# Plotando sismogramas

V1, sigma1, rec1 = elastic_forward(model, src, rec, order=1, src_direction='x')
V2, sigma2, rec2 = elastic_forward(model, src, rec, order=1, src_direction='z')

In [ ]:
plot_options = {'extent':[0, model.domain_size[0], src.time_range.stop, 0], 'cmap':'Greys', 'aspect':'auto'}

fig, axes = plt.subplots(1, 3, figsize=(12,7), sharey=True)

amax = max(abs(rec[1].data.min()), rec[1].data.max(), abs(rec1[1].data.min()), rec1[1].data.max(), abs(rec2[0].data.min()), rec2[0].data.max()) * .5
axes[0].imshow(rec[1].data, **plot_options, vmin=-amax, vmax=amax)
axes[0].set_title('Fonte: $\\sigma$ \n$v_{zz}$')
axes[0].set_xlabel('Distância (m)')

axes[1].imshow(rec1[1].data, **plot_options, vmin=-amax, vmax=amax)
axes[1].set_title('Fonte: $\\sigma_x$ \n$v_{zz}$')
axes[1].set_xlabel('Distância (m)')

axes[2].imshow(rec2[0].data, **plot_options, vmin=-amax, vmax=amax)
axes[2].set_title('Fonte: $\\sigma_z$ \n$v_{xx}$')
axes[2].set_xlabel('Distância (m)')

fig.tight_layout()
plt.show()

In [ ]:
snaps = np.linspace(0, (src.time_range.num - 1) // 2, 6, dtype='int16')

amax0 = max(V[1].data.max(), abs(V[1].data.min())) / 1e2
amax1 = max(V1[1].data.max(), abs(V1[1].data.min())) / 1e5
amax2 = max(V2[0].data.max(), abs(V2[0].data.min())) / 1e5
plot_options = {'cmap':'gray'}#, 'extent':[model.domain_size[0], 0, model.domain_size[1], 0]}

fig, axes = plt.subplots(3, 6, figsize=(20, 7), sharex=True, sharey=True)

[axes[0,i].imshow(V[1].data[snap].T, **plot_options, vmin=-amax0, vmax=amax0) for i, snap in enumerate(snaps)]
[axes[1,i].imshow(V1[1].data[snap].T, **plot_options, vmin=-amax1, vmax=amax2) for i, snap in enumerate(snaps)]
[axes[2,i].imshow(V2[0].data[snap].T, **plot_options, vmin=-amax2, vmax=amax2) for i, snap in enumerate(snaps)]
axes[0,0].set_ylabel('Campo $V0_z$ ($\\sigma$) \n\nProfundidade (m)')
axes[1,0].set_ylabel('Campo $V0_z$ ($\\sigma_{xx}$) \n\nProfundidade (m)')
axes[2,0].set_ylabel('Campo $V0_x$ ($\\sigma_{zz}$) \n\nProfundidade (m)')
[axes[2,i].set_xlabel('Distância (m)') for i in range(6)]
[axes[0,i].set_title(f'Snapshot: {round(src.time_values[snap], 2)}ms') for i, snap in enumerate(snaps)]

fig.tight_layout()
plt.show()

In [ ]:
V0, sigma0, rec0 = elastic_forward(model, src, rec, order=1, src_direction=None)
V1, sigma1, rec1 = elastic_forward(model, src, rec, order=1, src_direction='x')
V2, sigma2, rec2 = elastic_forward(model, src, rec, order=1, src_direction='z')
V3, sigma3, rec3 = elastic_forward(model, src, rec, order=1, src_direction='c')

In [ ]:
plot_matrix = np.asarray([[V0[0], V0[1], sigma0[0], sigma0[1], sigma0[2]],
                        [V1[0], V1[1], sigma1[0], sigma1[1], sigma1[2]],
                        [V2[0], V2[1], sigma2[0], sigma2[1], sigma2[2]],
                        [V3[0], V3[1], sigma3[0], sigma3[1], sigma3[2]]])

plot_options = {'cmap':'gray'}#, 'extent':[model.domain_size[0], 0, model.domain_size[1], 0]}

rows, cols = plot_matrix.shape[0], plot_matrix.shape[1]
snap = 300

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), sharex=True, sharey=True)

[[axes[row, col].imshow(plot_matrix[row,col].data[snap].T, vmin=plot_matrix[row,col].data.min(), vmax=plot_matrix[row,col].data.max(), **plot_options) for row in range(rows)] for col in range(cols)]

fig.tight_layout()
plt.show()

## a), b) e c)

Analisando o sismograma das velocidade vertical do experimento 6 (do meio), é possível perceber os receptores imediatamente acima da fonte tem dificuldade em registrar alguma perturbação.
Isso ocorre por conta da deformação imposta pela fonte. A fonte, nesse experimento, deformou a Terra ao longo do eixo horizontal. Com isso, a onda se propagou de forma mais efetiva na mesma direção da perturbação da fonte,
tento pouco efeito na direção vertical.

Comparando os registros dos experimentos 6(a) (do meio) e 6(b) (da direita), observa-se também uma redução na amplitude da onda registrada em receptores imediatamente acima da fonte. Isso ocorreu devido ao mesmo motivo do experimento anterior: a diferença entre as direções de perturbação provocada pela fonte sísmica e do registro da velocidade de partícula.

Além disso, o registro dos receptores do último experimento apresenta uma troca de polaridade entre as amplitudes de uma mesma frente de onda. Isso ocorre devido à direção de registro. Como os receptores registram perturbações na horizontal, as perturbações provocadas por uma frente de onda que caminha da direita para a equerda tem um sentido contrário àquelas provocadas por uma frente de onda que caminha da esquerda para a direita.

In [10]:
# --------- FIM --------- #